In [20]:
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os

# =========================
# FIX SEED FOR REPRODUCIBILITY
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 123
set_seed(SEED)

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = {"inaction": 0, "move": 1, "work": 2}
CONFIG = {
    "data_root": "/kaggle/input/datasets/iowiqo/lab6-embeds/embeddings",
    "max_windows_per_track": {"inaction": 0, "move": 1, "work": 0},
    "hidden": 448,                # число каналов в TCN (и в Transformer)
    "dropout": 0.3,
    "lr": 1e-3,
    "epochs": 50,
    "patience": 12,
    "label_smoothing": 0.05,
    "weight_decay": 2e-3,
    "batch_size": 32,
    "seed": SEED,
    "tcn_layers": 2,              # количество остаточных блоков TCN
    "kernel_size": 3,             # размер ядра TCN
    "save_dir": "./models",       # директория для сохранения моделей
}

# =========================
# DATA LOADING
# =========================
def load_sequence(folder: Path):
    frames = sorted(folder.glob("*.npy"))[:8]
    return np.stack([np.load(f) for f in frames])

def sample_windows_from_track(folder: Path, window: int = 8):
    frames = sorted(folder.glob("*.npy"))
    if len(frames) < window:
        return []
    return [frames[i:i+window] for i in range(len(frames) - window + 1)]

def build_dataset(root: str, max_windows_per_track: dict):
    X, y, groups = [], [], []
    root = Path(root)

    for class_name, label in CLASSES.items():
        class_dir = root / class_name
        if not class_dir.exists():
            continue

        max_w = max_windows_per_track.get(class_name, 3)

        for folder in class_dir.iterdir():
            if not folder.is_dir():
                continue
            name = folder.name.lower()

            if name[0].isdigit():
                x = load_sequence(folder)
                X.append(x)
                y.append(label)
                groups.append(name)

            elif name.startswith("tr"):
                windows = sample_windows_from_track(folder, window=8)
                if not windows:
                    continue
                sampled = random.sample(windows, k=min(max_w, len(windows)))
                for w in sampled:
                    x = np.stack([np.load(f) for f in w])
                    X.append(x)
                    y.append(label)
                    groups.append(name)

    return np.array(X), np.array(y), np.array(groups)

# =========================
# DATASET
# =========================
class ActionDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x, y = self.X[idx], self.y[idx]
        if self.augment:
            if torch.rand(1).item() < 0.5:
                x = x + torch.randn_like(x) * 0.01
        return x, y

# =========================
# TCN BLOCK (остаётся без изменений)
# =========================
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

# =========================
# POSITIONAL ENCODING (нужен для Transformer)
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=8):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# =========================
# ГИБРИДНАЯ МОДЕЛЬ: TCN + Transformer
# =========================
class TCNTransformer(nn.Module):
    def __init__(self, input_dim=768, hidden=384, num_classes=3, dropout=0.4,
                 num_layers=3, kernel_size=3):
        super().__init__()
        # TCN часть – стек из TemporalBlock с дилатациями
        tcn_blocks = []
        in_channels = input_dim
        for i in range(num_layers):
            dilation = 2 ** i
            tcn_blocks.append(TemporalBlock(in_channels, hidden, kernel_size,
                                            dilation, dropout))
            in_channels = hidden
        self.tcn = nn.Sequential(*tcn_blocks)

        # Transformer часть
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden))
        self.pos_enc = PositionalEncoding(hidden, max_len=9)  # 8 кадров + 1 CLS
        # Выбираем число голов так, чтобы делилось на hidden
        nhead = 8 if hidden % 8 == 0 else 4
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=nhead,
            dim_feedforward=hidden * 2,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Классификатор
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, num_classes)
        )

    def forward(self, x):
        # x: [B, 8, 768] -> [B, 768, 8] для TCN
        x = x.transpose(1, 2)
        x = self.tcn(x)                     # [B, hidden, 8]
        x = x.transpose(1, 2)               # [B, 8, hidden]

        B = x.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # [B, 1, hidden]
        x = torch.cat([cls_tokens, x], dim=1)          # [B, 9, hidden]
        x = self.pos_enc(x)
        x = self.transformer(x)                        # [B, 9, hidden]
        cls_out = x[:, 0, :]                           # [B, hidden]
        return self.head(cls_out)

# =========================
# TRAINING UTILS
# =========================
def compute_class_weights(y: np.ndarray):
    counts = np.bincount(y)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(counts)
    return torch.tensor(weights, dtype=torch.float32)

def train_one_fold(model, train_loader, val_loader, optimizer, criterion, cfg, fold):
    epochs = cfg["epochs"]
    patience = cfg["patience"]

    scaler = torch.amp.GradScaler('cuda')
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                  patience=5, min_lr=1e-6)

    best_f1 = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(epochs):
        # -------- Train --------
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        # -------- Eval --------
        model.eval()
        all_preds, all_labels = [], []
        correct, total = 0, 0
        class_correct = [0, 0, 0]
        class_total = [0, 0, 0]

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                preds = logits.argmax(dim=1)

                correct += (preds == y).sum().item()
                total += y.size(0)

                for c in range(3):
                    mask = (y == c)
                    class_total[c] += mask.sum().item()
                    class_correct[c] += (preds[mask] == c).sum().item()

                all_preds.append(preds.cpu())
                all_labels.append(y.cpu())

        all_preds = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()

        acc = correct / total
        ina_acc = class_correct[0] / max(1, class_total[0])
        mov_acc = class_correct[1] / max(1, class_total[1])
        wrk_acc = class_correct[2] / max(1, class_total[2])
        f1 = f1_score(all_labels, all_preds, average='macro')

        avg_loss = total_loss / len(train_loader)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:2d}: loss={avg_loss:.4f}, acc={acc:.4f}, "
              f"f1={f1:.4f}, lr={current_lr:.2e} | "
              f"ina={ina_acc:.3f} mov={mov_acc:.3f} wrk={wrk_acc:.3f}")

        scheduler.step(f1)

        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict().copy()
            no_improve = 0
            # Сохраняем лучшую модель при каждом улучшении
            save_path = os.path.join(cfg["save_dir"], f"fold_{fold}_best.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_state,
                'best_f1': best_f1,
                'config': cfg
            }, save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return best_f1

# =========================
# MAIN CV
# =========================
def run_single_cv(X, y, groups, cfg):
    # Создаём директорию для сохранения моделей
    os.makedirs(cfg["save_dir"], exist_ok=True)
    
    train_dataset = ActionDataset(X, y, augment=True)
    val_dataset = ActionDataset(X, y, augment=False)

    gkf = GroupKFold(n_splits=5)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"\n===== FOLD {fold} =====")
        set_seed(cfg["seed"] + fold)

        train_ds = Subset(train_dataset, train_idx)
        val_ds = Subset(val_dataset, val_idx)

        g = torch.Generator()
        g.manual_seed(cfg["seed"] + fold)
        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                                  shuffle=True, generator=g)
        val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"],
                                shuffle=False)

        # Используем гибридную модель вместо чистой TCN
        model = TCNTransformer(
            input_dim=768,
            hidden=cfg["hidden"],
            num_classes=3,
            dropout=cfg["dropout"],
            num_layers=cfg["tcn_layers"],
            kernel_size=cfg["kernel_size"]
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"]
        )

        class_weights = compute_class_weights(y[train_idx]).to(DEVICE)
        criterion = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=cfg["label_smoothing"]
        )

        score = train_one_fold(model, train_loader, val_loader,
                               optimizer, criterion, cfg, fold)
        fold_scores.append(score)
        print(f"Fold {fold} best macro F1: {score:.4f}")
        print(f"Model saved to: {cfg['save_dir']}/fold_{fold}_best.pth")

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    print(f"\nCV results (macro F1): {mean_f1:.4f} ± {std_f1:.4f}")
    print(f"All models saved in: {cfg['save_dir']}/")
    return fold_scores

# =========================
# ЗАПУСК
# =========================
if __name__ == "__main__":
    X, y, groups = build_dataset(
        CONFIG["data_root"],
        CONFIG["max_windows_per_track"]
    )
    print(f"Total samples: {len(X)}, class distribution: {np.bincount(y)}")
    run_single_cv(X, y, groups, CONFIG)

Total samples: 456, class distribution: [ 68 150 238]

===== FOLD 0 =====
Epoch  0: loss=0.8029, acc=0.8587, f1=0.8163, lr=1.00e-03 | ina=0.643 mov=0.758 wrk=1.000
Epoch  1: loss=0.4592, acc=0.8370, f1=0.7764, lr=1.00e-03 | ina=0.571 mov=0.727 wrk=1.000
Epoch  2: loss=0.4233, acc=0.8913, f1=0.8577, lr=1.00e-03 | ina=0.857 mov=0.788 wrk=0.978
Epoch  3: loss=0.3717, acc=0.8696, f1=0.8330, lr=1.00e-03 | ina=0.714 mov=0.848 wrk=0.933
Epoch  4: loss=0.3681, acc=0.8913, f1=0.8504, lr=1.00e-03 | ina=0.643 mov=0.848 wrk=1.000
Epoch  5: loss=0.2960, acc=0.8913, f1=0.8441, lr=1.00e-03 | ina=0.714 mov=0.879 wrk=0.956
Epoch  6: loss=0.2804, acc=0.8696, f1=0.8319, lr=1.00e-03 | ina=0.643 mov=0.788 wrk=1.000
Epoch  7: loss=0.3749, acc=0.8804, f1=0.8486, lr=1.00e-03 | ina=0.786 mov=0.909 wrk=0.889
Epoch  8: loss=0.2893, acc=0.9022, f1=0.8700, lr=1.00e-03 | ina=0.714 mov=0.848 wrk=1.000
Epoch  9: loss=0.2855, acc=0.8804, f1=0.8290, lr=1.00e-03 | ina=0.571 mov=0.909 wrk=0.956
Epoch 10: loss=0.3590, acc

In [19]:
import shutil
shutil.make_archive('/kaggle/working/models', 'zip', '/kaggle/working/models')

'/kaggle/working/models.zip'